<a href="https://colab.research.google.com/github/manuellazarte/Procesamiento-del-Lenguaje-Natural/blob/main/src/Trabajo_pr%C3%A1ctico_N2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pgvector

In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("/content/libros.csv")

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   titulo            150 non-null    object
 1   autores           150 non-null    object
 2   generos           150 non-null    object
 3   serie             48 non-null     object
 4   sinopsis          150 non-null    object
 5   url_libro         150 non-null    object
 6   categoria_origen  150 non-null    object
 7   fecha_extraccion  150 non-null    object
 8   url_portada       150 non-null    object
dtypes: object(9)
memory usage: 10.7+ KB


In [9]:
df.head()

,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion,url_portada
0,Batallón disciplinario,Karl von Vereiter,Aventuras; Bélico; Novela,NaN,De todas las unidades especiales que la Wehrma...,https://ww3.lectulandia.com/book/batallon-disc...,belico,2026-09-01,https://assets.lectulandia.com/b/ab/Karl%20von...
1,Bombardero,Len Deighton,Bélico; Intriga; Novela,NaN,"En una calurosa noche de junio de 1943, una fu...",https://ww3.lectulandia.com/book/bombardero/,belico,2026-09-01,https://assets.lectulandia.com/b/ab/Len%20Deig...
2,Las quimeras negras,Jean Lartéguy,Bélico; Novela,NaN,En medio de un África presa de todos los delir...,https://ww3.lectulandia.com/book/las-quimeras-...,belico,2026-09-01,https://assets.lectulandia.com/b/ab/Jean%20Lar...
3,Tierra de asilo,Jean Lartéguy,Bélico; Novela,NaN,Sauveterre es un rincón de la Francia olvidada...,https://ww3.lectulandia.com/book/tierra-de-asilo/,belico,2026-09-01,https://assets.lectulandia.com/b/ab/Jean%20Lar...
4,Colina 112,Adrian Goldsworthy,Bélico; Histórico; Novela,NaN,"Se avecina el día D. Sin embargo, para tres jó...",https://ww3.lectulandia.com/book/colina-112/,belico,2026-09-01,https://assets.lectulandia.com/b/ab/Adrian%20G...


### Preprocesamiento

In [10]:
import unicodedata

# Definimos funcion para remover acentos
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])

In [11]:
# Definimos la lista de stopwords en español
stopwords_es = [
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un',
    'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le',
    'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin',
    'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo',
    'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso',
    'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo',
    'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes',
    'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas', 'algo',
    'nosotros', 'mi', 'mis', 'tú', 'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras',
    'vosotros', 'vosotras', 'os', 'mío', 'mía', 'míos', 'mías', 'tuyo', 'tuya',
    'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 'suyas', 'nuestro', 'nuestra',
    'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 'vuestras', 'es'
]

In [12]:
import re

# Definimos el preprocesador para aplicar la limpieza
def preprocesar_lexico(texto):
    texto = texto.lower() # minisculas
    texto = re.sub(r'[^\w\s]', '', texto) # puntuacion
    texto = remove_accents(texto) # acentos
    palabras = [w for w in texto.split() if w not in stopwords_es] # guardar stopwords
    return " ".join(palabras) # quitar stopwords

Limpieza del texto en crudo aplicada en una nueva columna limpia

In [15]:
df['sinopsis_limpia'] = df['sinopsis'].apply(preprocesar_lexico)

print("Sinopsis en crudo:")
print(df['sinopsis'])

print("\nSinopsis limpia:")
print(df['sinopsis_limpia'])

Sinopsis en crudo:
0      De todas las unidades especiales que la Wehrma...
1      En una calurosa noche de junio de 1943, una fu...
2      En medio de un África presa de todos los delir...
3      Sauveterre es un rincón de la Francia olvidada...
4      Se avecina el día D. Sin embargo, para tres jó...
                             ...                        
145    Alfa Ndiaye es senegalés y ha acabado combatie...
146    Veinte años después de la derrota del ejército...
147    Un día, una niña de apenas cinco años preguntó...
148    En esta sensacional novela sobre pilotos naval...
149    En la madrugada del 30 de abril de 1945, el ra...
Name: sinopsis, Length: 150, dtype: object

Sinopsis limpia:
0      todas unidades especiales wehrmacht poseyo tra...
1      calurosa noche junio 1943 fuerza 700 bombarder...
2      medio africa presa delirios tres hombres consu...
3      sauveterre rincon francia olvidada cerril auto...
4      avecina dia d embargo tres jovenes gales sur g...
        